In [0]:
from pyspark.sql import functions as F

In [0]:
BRONZE = "abfss://bronze@pharmadatalakeuk.dfs.core.windows.net"
SILVER = "abfss://silver@pharmadatalakeuk.dfs.core.windows.net"
lims = (spark.read
       .option("header", "true")
       .csv(f"{BRONZE}/lims/lims_results.csv"))


In [0]:
lims_dedup = lims.dropDuplicates()
print(f"Rows: {lims.count()}")
print(f"After :{lims_dedup.count()}")
#print(lims_dedup.select("batch_id").distinct().count())

Rows: 1027
After :1015


In [0]:
display(lims)

test_id,batch_id,test_name,unit,lower_spec_limit,upper_spec_limit,test_result,result_status,release_status,test_date,analyst_id
LT-000001,BATCH-00002,Assay,%,95.0,105.0,100.5,PASS,Released,2024-01-07,AN-015
LT-000002,BATCH-00002,pH,pH,5.5,7.5,6.71,PASS,Released,2024-01-07,AN-001
LT-000003,BATCH-00001,Assay,%,95.0,105.0,99.9,PASS,Released,2024-01-05,AN-024
LT-000004,BATCH-00001,Dissolution,%,80.0,100.0,92.3,PASS,Released,06/01/2024,AN-013
LT-000005,BATCH-00003,Assay,%,95.0,105.0,101.8,PASS,Released,2024-01-05,AN-021
LT-000006,BATCH-00003,Impurities,%,0.0,0.5,0.03,PASS,Released,2024-01-06,AN-025
LT-000007,BATCH-00005,Assay,%,95.0,105.0,99.8,PASS,Released,2024-01-07,AN-017
LT-000008,BATCH-00005,Impurities,%,0.0,0.5,0.166,PASS,Released,2024-01-08,AN-007
LT-000009,BATCH-00006,Assay,%,95.0,105.0,100.5,Pass,Released,06/01/2024,AN-022
LT-000010,BATCH-00006,Dissolution,%,80.0,100.0,92.0,PASS,Released,2024-01-08,AN-005


In [0]:
lims_trimmed =lims_dedup.select([F.trim(c).alias(c) for c in lims_dedup.columns]) 
display(lims_trimmed)

test_id,batch_id,test_name,unit,lower_spec_limit,upper_spec_limit,test_result,result_status,release_status,test_date,analyst_id
LT-000005,BATCH-00003,Assay,%,95.0,105.0,101.8,PASS,Released,2024-01-05,AN-021
LT-000012,BATCH-00004,Assay,%,95.0,105.0,103.9,PASS,Released,2024-01-06,AN-015
LT-000016,BATCH-00007,Content uniformity,%,90.0,110.0,null,PASS,Released,2024-01-07,AN-006
LT-000027,BATCH-00013,Content uniformity,%,90.0,110.0,96.8,PASS,Released,2024-01-11,AN-021
LT-000037,BATCH-00017,Content uniformity,%,90.0,110.0,97.8,PASS,released,2024-01-16,AN-007
LT-000039,BATCH-00018,Assay,%,95.0,105.0,93.6,OOS,Quarantine,2024-01-18,AN-022
LT-000118,BATCH-00057,Assay,%,95.0,105.0,99.6,PASS,Released,2024-02-17,AN-010
LT-000124,BATCH-00061,Assay,%,95.0,105.0,98.4,PASS,Quarantine,2024-02-19,AN-020
LT-000155,BATCH-00075,pH,pH,5.5,7.5,6.7,PASS,Released,2024-02-25,AN-006
LT-000159,BATCH-00078,Hardness,N,60.0,120.0,84.1,PASS,Released,2024-02-26,AN-023


In [0]:
# display(

#     lims_trimmed.groupBy("result_status").agg(F.count("result_status"))
# )

In [0]:
result_status_clean = (
                        F.when(F.upper(F.col("result_status")).isin("PASS","P","PASSED"),"Pass")
                        .when(F.upper(F.col("result_status")).isin("FAIL","F","FAILED"),"Fail")
                        .when(F.upper(F.col("result_status"))=="OOS","OOS")
                        .otherwise(F.lit(None))
)

In [0]:
lims_standardised = lims_trimmed.withColumn("batch_id",F.upper(F.col("batch_id")))\
                                .withColumn("result_status",result_status_clean)\
                                .withColumn("release_status",F.initcap(F.lower(F.col("release_status"))))



In [0]:
# display(
#     lims_standardised.groupBy("release_status").agg(F.count("*"))
# )

In [0]:
display(
    lims_standardised.groupBy("result_status").agg(F.count("*"))
)

result_status,count(1)
Pass,962
OOS,15
Fail,38


In [0]:
display(lims_standardised.withColumn
("testt",F.regexp_extract(F.col("test_result"), r"(-?\d+(?:\.\d+)?)", 1)))

test_id,batch_id,test_name,unit,lower_spec_limit,upper_spec_limit,test_result,result_status,release_status,test_date,analyst_id,testt
LT-000005,BATCH-00003,Assay,%,95.0,105.0,101.8,Pass,Released,2024-01-05,AN-021,101.8
LT-000012,BATCH-00004,Assay,%,95.0,105.0,103.9,Pass,Released,2024-01-06,AN-015,103.9
LT-000016,BATCH-00007,Content uniformity,%,90.0,110.0,null,Pass,Released,2024-01-07,AN-006,null
LT-000027,BATCH-00013,Content uniformity,%,90.0,110.0,96.8,Pass,Released,2024-01-11,AN-021,96.8
LT-000037,BATCH-00017,Content uniformity,%,90.0,110.0,97.8,Pass,Released,2024-01-16,AN-007,97.8
LT-000039,BATCH-00018,Assay,%,95.0,105.0,93.6,OOS,Quarantine,2024-01-18,AN-022,93.6
LT-000118,BATCH-00057,Assay,%,95.0,105.0,99.6,Pass,Released,2024-02-17,AN-010,99.6
LT-000124,BATCH-00061,Assay,%,95.0,105.0,98.4,Pass,Quarantine,2024-02-19,AN-020,98.4
LT-000155,BATCH-00075,pH,pH,5.5,7.5,6.7,Pass,Released,2024-02-25,AN-006,6.7
LT-000159,BATCH-00078,Hardness,N,60.0,120.0,84.1,Pass,Released,2024-02-26,AN-023,84.1


In [0]:
lims_typed = (lims_standardised.withColumn("below_reporting_limit",F.col("test_result").startswith("<"))
            .withColumn("test_result",F.regexp_extract("test_result",r"(-?\d+(?:\.\d+)?)" , 1).try_cast("double"))
            .withColumn("lower_spec_limit",F.col("lower_spec_limit").try_cast("double"))
            .withColumn("upper_spec_limit",F.col("upper_spec_limit").try_cast("double"))
            .withColumn("limits_swapped",F.when(F.col("lower_spec_limit")>F.col("upper_spec_limit"),True)
                        .otherwise(False))
            .withColumns({
                            "lower_spec_limit": F.when(F.col("limits_swapped"), F.col("upper_spec_limit"))
                                                .otherwise(F.col("lower_spec_limit")),
                            "upper_spec_limit": F.when(F.col("limits_swapped"), F.col("lower_spec_limit"))
                                                .otherwise(F.col("upper_spec_limit"))
                        })
            
            )

In [0]:
print("Below reporting limit:", lims_typed.filter(F.col("below_reporting_limit")).count())
print("Null test_result:     ", lims_typed.filter(F.col("test_result").isNull()).count())
print("Limits swapped:       ", lims_typed.filter(F.col("limits_swapped")).count())
print("Still reversed:       ", lims_typed.filter(F.col("lower_spec_limit") > F.col("upper_spec_limit")).count())
print("One limit missing:    ", lims_typed.filter(F.col("lower_spec_limit").isNull() | F.col("upper_spec_limit").isNull()).count())

Below reporting limit: 10
Null test_result:      17
Limits swapped:        6
Still reversed:        0
One limit missing:     8


In [0]:
display(lims_typed.select("test_date").distinct().limit(20))

test_date
2024-01-10
2024-01-11
2024-01-19
2024-01-22
2024-02-03
2024-02-20
28/02/2024
2024-03-06
13-Mar-2024
2024-03-25


In [0]:
def parse_date(column_name):
    col = F.col(column_name)
    return F.coalesce(
            F.try_to_timestamp(col,F.lit("yyyy-MM-dd")),
            F.try_to_timestamp(col,F.lit("dd/MM/yyyy")),
            F.try_to_timestamp(col,F.lit("dd-MMM-yyyy"))       
    ).cast("date")

In [0]:
lims_parsed = lims_typed.withColumn("test_date",parse_date("test_date"))
display(lims_parsed)

test_id,batch_id,test_name,unit,lower_spec_limit,upper_spec_limit,test_result,result_status,release_status,test_date,analyst_id,below_reporting_limit,limits_swapped
LT-000005,BATCH-00003,Assay,%,95.0,105.0,101.8,Pass,Released,2024-01-05,AN-021,false,false
LT-000012,BATCH-00004,Assay,%,95.0,105.0,103.9,Pass,Released,2024-01-06,AN-015,false,false
LT-000016,BATCH-00007,Content uniformity,%,90.0,110.0,null,Pass,Released,2024-01-07,AN-006,null,false
LT-000027,BATCH-00013,Content uniformity,%,90.0,110.0,96.8,Pass,Released,2024-01-11,AN-021,false,false
LT-000037,BATCH-00017,Content uniformity,%,90.0,110.0,97.8,Pass,Released,2024-01-16,AN-007,false,false
LT-000039,BATCH-00018,Assay,%,95.0,105.0,93.6,OOS,Quarantine,2024-01-18,AN-022,false,false
LT-000118,BATCH-00057,Assay,%,95.0,105.0,99.6,Pass,Released,2024-02-17,AN-010,false,false
LT-000124,BATCH-00061,Assay,%,95.0,105.0,98.4,Pass,Quarantine,2024-02-19,AN-020,false,false
LT-000155,BATCH-00075,pH,pH,5.5,7.5,6.7,Pass,Released,2024-02-25,AN-006,false,false
LT-000159,BATCH-00078,Hardness,N,60.0,120.0,84.1,Pass,Released,2024-02-26,AN-023,false,false


In [0]:
lims_parsed.printSchema()

root
 |-- test_id: string (nullable = true)
 |-- batch_id: string (nullable = true)
 |-- test_name: string (nullable = true)
 |-- unit: string (nullable = true)
 |-- lower_spec_limit: double (nullable = true)
 |-- upper_spec_limit: double (nullable = true)
 |-- test_result: double (nullable = true)
 |-- result_status: string (nullable = true)
 |-- release_status: string (nullable = true)
 |-- test_date: date (nullable = true)
 |-- analyst_id: string (nullable = true)
 |-- below_reporting_limit: boolean (nullable = true)
 |-- limits_swapped: boolean (nullable = false)



In [0]:
lims_final =(lims_parsed
    .withColumn("calculated_status",
                F.when((F.col("test_result").isNull())|(F.col("lower_spec_limit").isNull())|(F.col("upper_spec_limit").isNull()),F.lit(None))
                .when(((F.col("test_result"))>=F.col("lower_spec_limit")) &(F.col("test_result")<=(F.col("upper_spec_limit"))),"Pass")
                .otherwise("Fail") )       
    .withColumn("status_mismatch", F.when(((F.col("result_status")=="Pass") & (F.col("calculated_status")=="Fail"))|((F.col("result_status").isin("Fail","OOS"))&(F.col("calculated_status")=="Pass")),True)
    .otherwise(False))                         
)                         

In [0]:
display(lims_final.groupBy("status_mismatch").count())

status_mismatch,count
false,997
true,18


In [0]:
mes_batches = (spark.read.format("delta").load(f"{SILVER}/mes")
               .select("batch_id","start_time")
               .withColumnRenamed("start_time","batch_start_time")
               .withColumn("in_mes",F.lit(True))
               )
               

In [0]:
lims_silver= (lims_final.join(mes_batches, on="batch_id",how="left")
                .withColumn("is_orphan", F.col("in_mes").isNull())
                .drop("in_mes")
                .withColumn("test_before_batch_start",
                            F.when((F.col("test_date")) < (F.to_date(F.col("batch_start_time"))),True)
                            .otherwise(False))
                .drop("batch_start_time")
)

In [0]:
lims_silver.write.format("delta").mode("overwrite").save(f"{SILVER}/lims")

In [0]:
spark.read.format("delta").load(f"{SILVER}/lims").printSchema()

root
 |-- batch_id: string (nullable = true)
 |-- test_id: string (nullable = true)
 |-- test_name: string (nullable = true)
 |-- unit: string (nullable = true)
 |-- lower_spec_limit: double (nullable = true)
 |-- upper_spec_limit: double (nullable = true)
 |-- test_result: double (nullable = true)
 |-- result_status: string (nullable = true)
 |-- release_status: string (nullable = true)
 |-- test_date: date (nullable = true)
 |-- analyst_id: string (nullable = true)
 |-- below_reporting_limit: boolean (nullable = true)
 |-- limits_swapped: boolean (nullable = true)
 |-- calculated_status: string (nullable = true)
 |-- status_mismatch: boolean (nullable = true)
 |-- is_orphan: boolean (nullable = true)
 |-- test_before_batch_start: boolean (nullable = true)



In [0]:
(lims_silver.write
 .mode("overwrite")
 .parquet(f"{SILVER}/export/lims"))
 
 
display(lims_silver)


batch_id,test_id,test_name,unit,lower_spec_limit,upper_spec_limit,test_result,result_status,release_status,test_date,analyst_id,below_reporting_limit,limits_swapped,calculated_status,status_mismatch,is_orphan,test_before_batch_start
BATCH-00003,LT-000005,Assay,%,95.0,105.0,101.8,Pass,Released,2024-01-05,AN-021,false,false,Pass,false,false,false
BATCH-00004,LT-000012,Assay,%,95.0,105.0,103.9,Pass,Released,2024-01-06,AN-015,false,false,Pass,false,false,false
BATCH-00007,LT-000016,Content uniformity,%,90.0,110.0,null,Pass,Released,2024-01-07,AN-006,null,false,null,false,false,false
BATCH-00013,LT-000027,Content uniformity,%,90.0,110.0,96.8,Pass,Released,2024-01-11,AN-021,false,false,Pass,false,false,false
BATCH-00017,LT-000037,Content uniformity,%,90.0,110.0,97.8,Pass,Released,2024-01-16,AN-007,false,false,Pass,false,false,false
BATCH-00018,LT-000039,Assay,%,95.0,105.0,93.6,OOS,Quarantine,2024-01-18,AN-022,false,false,Fail,false,false,false
BATCH-00057,LT-000118,Assay,%,95.0,105.0,99.6,Pass,Released,2024-02-17,AN-010,false,false,Pass,false,false,false
BATCH-00061,LT-000124,Assay,%,95.0,105.0,98.4,Pass,Quarantine,2024-02-19,AN-020,false,false,Pass,false,false,false
BATCH-00075,LT-000155,pH,pH,5.5,7.5,6.7,Pass,Released,2024-02-25,AN-006,false,false,Pass,false,false,false
BATCH-00078,LT-000159,Hardness,N,60.0,120.0,84.1,Pass,Released,2024-02-26,AN-023,false,false,Pass,false,false,false
